In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 3, Finished, Available, Finished, False)

In [2]:
bronze_path = f"Files/Bronze/"
silver_path = f"Tables/Silver/"
gold_path = f"Tables/Gold/"

StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F

StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 5, Finished, Available, Finished, False)

In [4]:
bronze_tickets = spark.read.format('csv').option('header','true').load(bronze_path + 'tickets.csv')
display(bronze_tickets)

StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 01adf6fe-42c8-4ef1-8853-588445ea40c9)

In [5]:
#cast types


silver_tickets = bronze_tickets.withColumn("created_at",F.to_timestamp("created_at")) \
.withColumn("first_response_at",F.to_timestamp("first_response_at")) \
.withColumn("resolved_at",F.to_timestamp("resolved_at")) \
.withColumn("reopened_count", F.col("reopened_count").cast("int")) \
.withColumn("csat_score", F.col("csat_score").cast("int"))


display(silver_tickets)

silver_tickets.write.mode("overwrite").format("delta").save(silver_path + "tickets")
silver_tickets = spark.read.format("delta").load(silver_path + "tickets")

StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 855594ef-6482-48ae-8abd-07fe0ddf3071)

In [6]:
gold_tickets = silver_tickets.withColumn("frt_minutes",F.when(F.col('first_response_at').isNotNull(), F.round(F.col('first_response_at').cast("long") -     \
F.col('created_at').cast("long"),0) / 60).otherwise(None)) \
.withColumn("ttr_minutes",F.when(F.col("resolved_at").isNotNull(), F.round(F.col("resolved_at").cast("long") - F.col("created_at").cast("long"),0)/60)  \
.otherwise(None))     

gold_tickets = gold_tickets.select(
F.col("ticket_id").alias("TicketId"),
F.col("created_at").alias("CreatedAt"),
F.col("first_response_at").alias("FirstResponseAt"),
F.col("resolved_at").alias("ResolvedAt"),
F.col("status").alias("Status"),
F.col("frt_minutes").alias("FirstResponseTimeMinutes"),
F.col("ttr_minutes").alias("TimeToResolveMinutes"),
F.col("reopened_count").alias("ReopenedCount"),
F.col("csat_score").alias("CustomerSatisfactionScore"),
F.col("channel").alias("Channel"),
F.col("customer_id").alias("CustomerId").cast("int"),
F.col("product_id").alias("ProductId").cast("int"),
F.col("agent_id").alias("AgentId").cast("int"),
F.col("sla_id").alias("SlaId").cast("int"),
F.col("breach_first_response").alias("BreachFirstResponse"),
F.col("breach_resolution").alias("BreachResolution"),
F.col("subject").alias("Subject")





)


gold_tickets.write.mode("overwrite").format("delta").option("overwriteSchema","true").save(gold_path + "Fact_tickets")


StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 8, Finished, Available, Finished, False)

In [7]:
#getting other tables from bronze to silver

customers_bz = spark.read.format('csv').option('header', 'true').load(bronze_path + 'customers.csv')


silver_customers = customers_bz.withColumn("customer_id", F.col("customer_id").cast("int")) \
.withColumn("mrr", F.col("mrr").cast("double")) \
.withColumn("is_vip", F.col("is_vip").cast("int")) \
.withColumn("created_date", F.to_date("created_date"))


silver_customers.write.mode("overwrite").format("delta").save(silver_path + "silver_customers")

StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 9, Finished, Available, Finished, False)

In [8]:
#agents
agents_bz = spark.read.format("csv").option("header","true").load("Files/Bronze/agents.csv")

silver_agents = agents_bz.withColumn("agent_id", F.col("agent_id").cast("int")) \
.withColumn("hire_date", F.to_date("hire_date"))

silver_agents.write.mode("overwrite").format("delta").save(silver_path + "silver_agents")

#products
products_bz = spark.read.format("csv").option("header","true").load(bronze_path + 'products.csv')
silver_products = products_bz.withColumn("product_id", F.col("product_id").cast("int"))
silver_products.write.mode("overwrite").format("delta").save(silver_path + "silver_products")



#channels

channels_bz = spark.read.format("csv").option("header","true").load(bronze_path + "channels.csv")
silver_channels = channels_bz.withColumn("channel_id", F.col("channel_id").cast("int"))
silver_channels.write.mode("overwrite").format("delta").save(silver_path + "silver_channels")



#sla_policies

policies_bz = spark.read.format("csv").option("header", "true").load(bronze_path + "sla_policies.csv")
silver_sla = policies_bz.withColumn("sla_id", F.col("sla_id").cast("int"))
silver_sla.write.mode("overwrite").format("delta").save(silver_path + "silver_policies")



#ticket_Comments

comments_bz = spark.read.format("csv").option("header", "true").load(bronze_path + "ticket_comments.csv")
silver_comments = comments_bz.withColumn("comment_id", F.col("comment_id").cast("int")) \
    .withColumn("ticket_id", F.col("ticket_id").cast("int")) \
    .withColumn("sentiment_score", F.col("sentiment_score").cast("double")) \
    .withColumn("comment_created_at", F.to_timestamp("comment_created_at"))

silver_comments.write.mode("overwrite").format("delta").save(silver_path + "silver_comments")



StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 10, Finished, Available, Finished, True)

In [9]:
#silver to gold dimensions

from pyspark.sql import functions as F, types as T
import datetime as dt


silver_cust = spark.read.format("delta").load(silver_path + "silver_customers")

dim_customer = silver_cust.select( 
    F.col("customer_id").alias("CustomerId"),
    F.col("account_name").alias("AccountName"),
    F.col("segment").alias("Segment"),
    F.col("region").alias("Region"),
    F.col("mrr").alias("MRR"),
    F.col("is_vip").alias("IsVIP"),
    F.col("created_date").alias("CreatedDate")
) .dropDuplicates(["CustomerId"]).withColumn("IsActive",F.lit(1)).withColumn("LoadDate",F.current_timestamp())

#IsActive flag =1 is for SCD 1 type


dim_customer.write.mode("overwrite").format("delta").save(gold_path + "gold_dim_customer")


#Dim agent

silver_agents = spark.read.format("delta").load(silver_path + "silver_agents")
dim_agents = silver_agents.select(
    F.col("agent_id").alias("AgentId"),
    F.col("agent_name").alias("AgentName"),
    F.col("manager_id").alias("ManagerId"),
    F.col("team").alias("Team"),
    F.col("region").alias("Region"),
    F.col("hire_date").alias("HireDate"),
    F.col("seniority").alias("Seniority")
).dropDuplicates(["AgentId"]).withColumn("IsActive", F.lit(1)).withColumn("LoadDate", F.current_timestamp())

dim_agents.write.mode("overwrite").format("delta").save(gold_path + "gold_dim_agents")

#dim product


silver_product = spark.read.format("delta").load(silver_path + "silver_products") 

dim_product = silver_product.select(
    F.col("product_id").alias("ProductId"),
    F.col("product_name").alias("ProductName"),
    F.col("category").alias("Category"),
    F.col("tier").alias("Tier")
).dropDuplicates(["ProductId"]).withColumn("IsActive", F.lit(1)).withColumn("LoadDate", F.current_timestamp())


dim_product.write.mode("overwrite").format("delta").save(gold_path + "gold_dim_product")


#Dim channel

silver_channels = spark.read.format("delta").load(silver_path + "silver_channels")
dim_channel = silver_channels.select(
    F.col("channel_id").alias("ChannelId"),
    F.col("channel_name").alias("Channel_Name")
).withColumn("IsActive", F.lit(1)).withColumn("LoadDate",F.current_timestamp())

dim_channel.write.mode("overwrite").format("delta").save(gold_path + "gold_dim_channel")



StatementMeta(, 49d2963d-21d5-4291-b9de-84a699b1fbdf, 11, Submitted, Running, Running, True)

In [ ]:
#Dim Date : Generated table : Builds a continuous date table i.e. calendar with all dates from min to max date in tickets dataset


fact_min = spark.read.format("delta").load(gold_path + "Fact_tickets").agg(F.min("CreatedAt")).first()[0]
fact_max = spark.read.format("delta").load(gold_path + "Fact_tickets").agg(F.max("CreatedAt")).first()[0]


start_date = fact_min.date()
end_date = fact_max.date() + dt.timedelta(days=365)

def daterange(start, end):
    for n in range((end-start).days + 1):
        yield start + dt.timedelta(n)

rows =[]


for d in daterange(start_date,end_date):
    rows.append((
        d,
        int(d.strftime("%Y%m%d")),
        d.year,
        ((d.month-1)//3)+1,
        d.month,
        d.day,
        d.strftime("%b"),
        d.strftime("%A"),
        int(d.strftime("%W")),
        int(d.strftime("%u"))
    ))






StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#schema definition


schema = T.StructType([
    T.StructField("Date", T.DateType()),
    T.StructField("DateKey", T.IntegerType()),
    T.StructField("Year", T.IntegerType()),
    T.StructField("Quarter", T.IntegerType()),
    T.StructField("Month", T.IntegerType()),
    T.StructField("Day", T.IntegerType()),
    T.StructField("MonthName", T.StringType()),
    T.StructField("DayName", T.StringType()),
    T.StructField("WeekOfYear", T.IntegerType()),
    T.StructField("DayOfWeek", T.IntegerType()),
])


dim_date = spark.createDataFrame(rows, schema)

dim_date.write.mode("overwrite").format("delta").save(gold_path + "gold_dim_date")




StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
fact = spark.read.format("delta").load(gold_path + "Fact_tickets")

fact_with_keys = fact.withColumn("CreatedDateKey", F.date_format("CreatedAt","yyyyMMdd").cast("int")) \
.withColumn("FirstResponseKey", F.when(F.col("FirstResponseAt").isNotNull(),  \
F.date_format("FirstResponseAt", "yyyyMMdd").cast("int")).otherwise(None))  \
.withColumn("ResolvedDateKey", F.when(F.col("ResolvedAt").isNotNull(),F.date_format("ResolvedAt","yyyyMMdd").cast("int")).otherwise(None))

fact_with_keys.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(gold_path + "Fact_tickets")



StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
df = spark.sql("SELECT * FROM Data.Gold.Fact_tickets LIMIT 1000")
display(df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
silver_sla = spark.read.format("delta").load(silver_path + "silver_policies")

dim_sla = silver_sla.select(
    F.col("sla_id").alias("SlaId"),
    F.col("priority").alias("Priority"),
    F.col("first_response_hours").alias("FirstResponseHours").cast("int"),
    F.col("resolution_hours").alias("ResolutionHours").cast("int")
)


dim_sla.write.mode("overwrite").format("delta").save(gold_path + "gold_dim_sla")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
def show_count(table):
    print(table, spark.read.format("delta").load(gold_path + table ).count())



for t in ["Fact_tickets","gold_dim_customer","gold_dim_agents","gold_dim_product","gold_dim_channel","gold_dim_date","gold_dim_sla"]:
    show_count(t)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#IDentify ORphan keys - validation of data

#Customers

spark.sql("USE Data.Gold")

orphan_cust = spark.sql("""
SELECT count(*) as Orphans
from Fact_tickets as f
left join gold_dim_customer as d
on f.CustomerId = d.CustomerId
where d.CustomerId is null""").first()[0]


print(f"Orphan Customers: {orphan_cust}")



orphan_agents = spark.sql(""" 
 select count(*) as orphans
 from Fact_tickets as f
 left join gold_dim_agents as d
 on f.AgentId = d.AgentId
 where d.AgentId is null
""").first()[0]
print(f"Orphan Agents: {orphan_agents}")


orphan_product = spark.sql("""
SELECT count(*) as Orphans
from Fact_tickets as f
left join gold_dim_product as d
on f.ProductId = d.ProductId
where d.ProductId is null""").first()[0]


print(f"Orphan Products: {orphan_product}")

orphan_channel = spark.sql("""
SELECT count(*) as Orphans
from Fact_tickets as f
left join gold_dim_channel as d
on f.Channel = d.Channel_Name
where d.Channel_Name is null
""").first()[0]
print(f"Orphan Channels: {orphan_channel}") 


orphan_date = spark.sql("""
SELECT count(*) as Orphans
from Fact_tickets as f
left join gold_dim_date as d
on f.CreatedDateKey = d.DateKey
where d.DateKey is null""").first()[0]
print(f"Orphan Dates: {orphan_date}")






StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#performance tuning

spark.sql("use Data.Gold")

fact_partitioned = spark.table("Fact_tickets").withColumn("year_month",F.date_format("CreatedAt","yyyyMM"))




StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
fact_partitioned.write.mode("overwrite").partitionBy("year_month").format("delta").option("overwriteSchema","true").save(gold_path + "Fact_tickets")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
df = spark.sql("SELECT * FROM Data.Gold.Fact_tickets LIMIT 1000")
display(df)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
spark.sql(""" 
OPTIMIZE Data.Gold.Fact_tickets ZORDER BY (CreatedAt)
""")

StatementMeta(, , -1, Waiting, , Waiting, True)